# Stage 03 - Gold Test Products

Publish traceable readiness, objective, data-quality, finding, and observed-versus-predicted products for the Direct Lake semantic model and report.

In [ ]:
from pyspark.sql import Window, functions as F

silver_baseline_df = spark.table("silver_emulated_test_baseline")
silver_observed_df = spark.table("silver_observed_events")
silver_operations_df = spark.table("silver_operations_snapshot")
silver_required_feed_checks_df = spark.table("silver_required_feed_checks")
silver_quarantine_df = spark.table("silver_quarantine_records")

OBJECTIVE_NAMES = {
    "obj-planning-readiness-001": "Confirm all participants are ready for execution",
    "obj-planning-alignment-001": "Verify timing and clock alignment",
    "obj-planning-observe-001": "Validate collection and observation coverage",
    "obj-planning-share-001": "Confirm data can be shared for analysis",
    "obj-planning-sustainment-001": "Confirm sustainment status is available",
}
PARTICIPANT_NAMES = {
    "army-int-bravo-01": "Army integration node Bravo",
    "bm-alpha-01": "Battle-management node Alpha",
    "patriot-bravo-01": "Patriot participant Bravo 01",
    "patriot-bravo-02": "Patriot participant Bravo 02",
    "patriot-alpha-01": "Patriot participant Alpha 01",
    "patriot-charlie-03": "Patriot participant Charlie 03",
    "thaad-alpha-01": "THAAD participant Alpha",
    "thaad-charlie-01": "THAAD participant Charlie",
    "tpy2-alpha-01": "TPY-2 sensor Alpha",
    "all-participants": "All test participants",
}
FEED_NAMES = {
    "feed-observed": "Observed test events",
    "feed-operations": "Operations readiness snapshot",
    "feed-command-integration": "Command integration events",
    "feed-sensor-observation": "Sensor observations",
    "feed-system-status": "Participant status",
    "feed-sustainment": "Sustainment status",
    "setup-summary": "Integrated setup summary",
}

def mapped_label(column_name, labels):
    mapping = F.create_map(*[value for pair in labels.items() for value in (F.lit(pair[0]), F.lit(pair[1]))])
    return F.coalesce(mapping[F.col(column_name)], F.col(column_name))

def add_report_labels(frame):
    if "test_objective_id" in frame.columns:
        frame = frame.withColumn("test_objective", mapped_label("test_objective_id", OBJECTIVE_NAMES))
    if "system_instance_id" in frame.columns:
        frame = frame.withColumn("participant", mapped_label("system_instance_id", PARTICIPANT_NAMES))
    if "required_feed_id" in frame.columns:
        frame = frame.withColumn("data_feed", mapped_label("required_feed_id", FEED_NAMES))
    if "comparison_status" in frame.columns:
        frame = frame.withColumn(
            "comparison_result",
            F.when(F.col("comparison_status") == "ON_PLAN", F.lit("On plan"))
            .when(F.col("comparison_status") == "PARTIAL", F.lit("Needs review"))
            .otherwise(F.lit("Not met")),
        )
    if "objective_status" in frame.columns:
        frame = frame.withColumn(
            "objective_result",
            F.when(F.col("objective_status") == "MET", F.lit("Met"))
            .when(F.col("objective_status") == "PARTIAL", F.lit("Needs review"))
            .otherwise(F.lit("Not met")),
        )
    if "required_feed_status" in frame.columns:
        frame = frame.withColumn(
            "feed_readiness",
            F.when(F.col("required_feed_status") == "MISSING", F.lit("Missing"))
            .when(F.col("required_feed_status") == "AVAILABLE", F.lit("Ready"))
            .otherwise(F.lit("See participant status")),
        )
    if "integrated_readiness_status" in frame.columns:
        frame = frame.withColumn(
            "readiness",
            F.when(F.col("integrated_readiness_status") == "READY", F.lit("Ready")).otherwise(F.lit("Limited")),
        )
    return frame

In [ ]:
gold_data_source_readiness_df = (
    silver_required_feed_checks_df
    .select(
        "scenario_id",
        "test_event_id",
        "test_objective_id",
        "required_feed_id",
        "system_instance_id",
        "required_feed_status",
        "setup_check_result",
        "snapshot_record_count",
        "valid_observed_count",
        "late_record_count",
        "notes",
    )
    .withColumn(
        "lesson_stand_in",
        F.when(F.col("required_feed_id") == "feed-sustainment", F.lit("Checked-in files intentionally omit the sustainment snapshot lane."))
        .otherwise(F.lit("Checked-in files stand in for live setup transports in this lesson."))
    )
)

gold_dry_run_discrepancies_df = (
    silver_operations_df
    .filter((F.col("record_type") == "READINESS_POLL") & (F.col("reported_state") != F.col("expected_state")))
    .select(
        "snapshot_record_id",
        "scenario_id",
        "test_event_id",
        "site_id",
        "system_instance_id",
        "test_objective_id",
        "reported_time_utc",
        "reported_state",
        "expected_state",
        "measure_name",
        "normalized_measure_value",
        "normalized_measure_unit",
        "notes",
    )
)

observed_actuals_df = silver_observed_df.select(
    "scenario_id",
    "test_event_id",
    "required_feed_id",
    "comparison_key",
    F.col("event_id").alias("actual_record_id"),
    F.col("event_type").alias("actual_source_type"),
    F.col("source_instance_id").alias("actual_system_instance_id"),
    F.col("site_id").alias("actual_site_id"),
    "actual_state",
    F.col("normalized_measure_value").alias("actual_value"),
    F.col("normalized_measure_unit").alias("actual_unit"),
    F.col("event_time_ts").alias("actual_time_utc"),
    "is_late_record",
    F.lit(None).cast("string").alias("actual_notes"),
)

feed_health_actuals_df = silver_operations_df.filter(F.col("record_type") == "FEED_HEALTH").select(
    "scenario_id",
    "test_event_id",
    "required_feed_id",
    "comparison_key",
    F.col("snapshot_record_id").alias("actual_record_id"),
    F.lit("operations.snapshot").alias("actual_source_type"),
    F.col("system_instance_id").alias("actual_system_instance_id"),
    F.col("site_id").alias("actual_site_id"),
    F.col("reported_state").alias("actual_state"),
    F.col("normalized_measure_value").alias("actual_value"),
    F.col("normalized_measure_unit").alias("actual_unit"),
    F.col("reported_time_utc").alias("actual_time_utc"),
    F.lit(False).alias("is_late_record"),
    F.col("notes").alias("actual_notes"),
)

all_actuals_df = observed_actuals_df.unionByName(feed_health_actuals_df, allowMissingColumns=True)

gold_observed_vs_predicted_df = (
    silver_baseline_df.alias("plan")
    .join(
        all_actuals_df.alias("actual"),
        ["scenario_id", "test_event_id", "required_feed_id", "comparison_key"],
        "left",
    )
    .withColumn("time_variance_seconds", F.unix_timestamp("actual_time_utc") - F.unix_timestamp("planned_event_time_ts"))
    .withColumn("numeric_delta", F.col("actual_value") - F.col("normalized_predicted_value"))
    .withColumn(
        "comparison_status",
        F.when(F.col("actual_record_id").isNull(), F.lit("NOT_MET"))
        .when(
            F.col("comparison_kind") == "processing_delay",
            F.when(F.col("actual_value") > F.col("normalized_predicted_value") * F.lit(2.0), F.lit("NOT_MET"))
            .when((F.col("actual_value") > F.col("normalized_predicted_value")) | F.col("is_late_record"), F.lit("PARTIAL"))
            .otherwise(F.lit("ON_PLAN"))
        )
        .when(
            F.col("comparison_kind").isin("health_score", "quality_score"),
            F.when(F.col("actual_state") != F.col("expected_state"), F.lit("PARTIAL"))
            .when(F.col("actual_value") < F.col("normalized_predicted_value"), F.lit("PARTIAL"))
            .otherwise(F.lit("ON_PLAN"))
        )
        .when(
            F.col("comparison_kind") == "feed_presence",
            F.when(F.col("actual_state") != F.col("expected_state"), F.lit("NOT_MET"))
            .otherwise(F.lit("ON_PLAN"))
        )
        .otherwise(F.when(F.col("actual_state") == F.col("expected_state"), F.lit("ON_PLAN")).otherwise(F.lit("PARTIAL")))
    )
    .select(
        "scenario_id",
        "test_event_id",
        "test_objective_id",
        "required_feed_id",
        "planned_record_id",
        "planned_event_type",
        "site_id",
        "system_instance_id",
        "track_id",
        "expected_state",
        "normalized_predicted_value",
        "normalized_predicted_unit",
        "actual_record_id",
        "actual_source_type",
        "actual_state",
        "actual_value",
        "actual_unit",
        "time_variance_seconds",
        "numeric_delta",
        "is_late_record",
        "comparison_status",
        "actual_notes",
    )
)

print(f"Gold readiness rows: {gold_data_source_readiness_df.count()}")
print(f"Gold comparison rows: {gold_observed_vs_predicted_df.count()}")

In [ ]:
gold_test_objective_status_df = (
    gold_observed_vs_predicted_df
    .groupBy("scenario_id", "test_event_id", "test_objective_id")
    .agg(
        F.count("*").alias("planned_checks"),
        F.sum(F.when(F.col("comparison_status") == "ON_PLAN", 1).otherwise(0)).alias("on_plan_checks"),
        F.sum(F.when(F.col("comparison_status") == "PARTIAL", 1).otherwise(0)).alias("partial_checks"),
        F.sum(F.when(F.col("comparison_status") == "NOT_MET", 1).otherwise(0)).alias("not_met_checks"),
        F.sum(F.when(F.col("is_late_record"), 1).otherwise(0)).alias("late_evidence_count"),
    )
    .withColumn(
        "objective_status",
        F.when((F.col("not_met_checks") > 0) & (F.col("on_plan_checks") == 0) & (F.col("partial_checks") == 0), F.lit("NOT_MET"))
        .when((F.col("not_met_checks") == 0) & (F.col("partial_checks") == 0), F.lit("MET"))
        .otherwise(F.lit("PARTIAL"))
    )
    .withColumn(
        "objective_note",
        F.when(F.col("objective_status") == "MET", F.lit("Observed data matched the planned expectation."))
        .when(F.col("objective_status") == "PARTIAL", F.lit("Observed data stayed traceable, but at least one check deviated or arrived late."))
        .otherwise(F.lit("The required feed or comparison outcome did not meet the recorded setup objective."))
    )
)

In [ ]:
status_window = Window.partitionBy("source_instance_id").orderBy(F.col("event_time_ts").desc_nulls_last())
poll_window = Window.partitionBy("system_instance_id").orderBy(F.col("reported_time_utc").desc_nulls_last())

latest_status_df = (
    silver_observed_df.filter(F.col("event_type") == "system.status")
    .withColumn("status_rank", F.row_number().over(status_window))
    .filter(F.col("status_rank") == 1)
    .select(
        "scenario_id",
        "test_event_id",
        "site_id",
        F.col("source_instance_id").alias("system_instance_id"),
        F.col("actual_state").alias("observed_readiness_state"),
        F.col("normalized_measure_value").alias("observed_health_score"),
        F.array_join(F.col("contributing_conditions"), "; ").alias("status_conditions"),
    )
)

latest_poll_df = (
    silver_operations_df.filter(F.col("record_type") == "READINESS_POLL")
    .withColumn("poll_rank", F.row_number().over(poll_window))
    .filter(F.col("poll_rank") == 1)
    .select(
        "scenario_id",
        "test_event_id",
        "site_id",
        "system_instance_id",
        F.col("reported_state").alias("operations_poll_state"),
        "normalized_measure_value",
        "normalized_measure_unit",
        "notes",
    )
)

system_reference_df = latest_status_df.select("scenario_id", "test_event_id", "site_id", "system_instance_id").unionByName(
    latest_poll_df.select("scenario_id", "test_event_id", "site_id", "system_instance_id"),
    allowMissingColumns=True,
).dropDuplicates(["scenario_id", "test_event_id", "system_instance_id"])

system_feed_df = (
    gold_data_source_readiness_df
    .groupBy("system_instance_id")
    .agg(
        F.array_join(F.collect_set("required_feed_id"), "; ").alias("required_feed_ids"),
        F.max(F.when(F.col("required_feed_status") == "MISSING", F.lit(1)).otherwise(F.lit(0))).alias("has_missing_feed"),
        F.array_join(F.collect_set(F.when(F.col("required_feed_status") == "MISSING", F.col("required_feed_id"))), "; ").alias("missing_feed_ids"),
    )
)

system_readiness_df = (
    system_reference_df
    .join(latest_status_df, ["scenario_id", "test_event_id", "site_id", "system_instance_id"], "left")
    .join(latest_poll_df, ["scenario_id", "test_event_id", "site_id", "system_instance_id"], "left")
    .join(system_feed_df, ["system_instance_id"], "left")
    .withColumn(
        "required_feed_status",
        F.when(F.col("has_missing_feed") == 1, F.lit("MISSING")).otherwise(F.lit("AVAILABLE_OR_NA"))
    )
    .withColumn(
        "integrated_readiness_status",
        F.when((F.col("required_feed_status") == "MISSING") | (F.col("observed_readiness_state") == "LIMITED") | (F.col("operations_poll_state") == "LIMITED"), F.lit("LIMITED")).otherwise(F.lit("READY"))
    )
    .withColumn(
        "blocking_reason",
        F.concat_ws("; ",
            F.when(F.col("observed_readiness_state") == "LIMITED", F.lit("Observed system status remained LIMITED.")),
            F.when(F.col("operations_poll_state") == "LIMITED", F.concat(F.lit("Readiness poll deviation: "), F.col("notes"))),
            F.when(F.col("required_feed_status") == "MISSING", F.concat(F.lit("Missing required feed: "), F.col("missing_feed_ids"))),
            F.col("status_conditions"),
        )
    )
    .select(
        F.lit("SYSTEM").alias("readiness_scope"),
        "scenario_id",
        "test_event_id",
        "site_id",
        "system_instance_id",
        "observed_readiness_state",
        "operations_poll_state",
        "observed_health_score",
        F.col("required_feed_ids").alias("required_feed_id"),
        "required_feed_status",
        "integrated_readiness_status",
        "blocking_reason",
    )
)

overall_readiness_df = (
    system_readiness_df.groupBy("scenario_id", "test_event_id")
    .agg(
        F.sum(F.when(F.col("integrated_readiness_status") != "READY", 1).otherwise(0)).alias("constrained_system_count"),
        F.array_join(F.collect_set("blocking_reason"), "; ").alias("blocking_reason"),
    )
    .withColumn("readiness_scope", F.lit("TEST_EVENT"))
    .withColumn("site_id", F.lit("all-sites"))
    .withColumn("system_instance_id", F.lit("all-participants"))
    .withColumn("observed_readiness_state", F.lit(None).cast("string"))
    .withColumn("operations_poll_state", F.lit(None).cast("string"))
    .withColumn("observed_health_score", F.lit(None).cast("double"))
    .withColumn("required_feed_id", F.lit("setup-summary"))
    .withColumn("required_feed_status", F.lit("SEE_SYSTEM_ROWS"))
    .withColumn(
        "integrated_readiness_status",
        F.when(F.col("constrained_system_count") > 0, F.lit("LIMITED")).otherwise(F.lit("READY"))
    )
    .drop("constrained_system_count")
    .select(system_readiness_df.columns)
)

gold_integrated_readiness_df = system_readiness_df.unionByName(overall_readiness_df)

In [ ]:
alignment_finding_df = gold_observed_vs_predicted_df.filter(F.col("test_objective_id") == "obj-planning-alignment-001").select(
    F.lit("finding-setup-alignment-001").alias("finding_id"),
    F.lit("Clock alignment review remained open after the recorded dry run.").alias("finding_title"),
    F.lit("OPEN").alias("finding_status"),
    "test_objective_id",
    F.lit("gold_observed_vs_predicted").alias("evidence_source_table"),
    F.col("actual_record_id").alias("evidence_record_id"),
    F.concat(F.lit("Observed processing delay = "), F.round(F.col("actual_value"), 2).cast("string"), F.lit(" "), F.col("actual_unit")).alias("evidence_summary"),
)

alignment_poll_finding_df = gold_dry_run_discrepancies_df.filter(F.col("system_instance_id") == "army-int-bravo-01").select(
    F.lit("finding-setup-alignment-001").alias("finding_id"),
    F.lit("Clock alignment review remained open after the recorded dry run.").alias("finding_title"),
    F.lit("OPEN").alias("finding_status"),
    "test_objective_id",
    F.lit("gold_dry_run_discrepancies").alias("evidence_source_table"),
    F.col("snapshot_record_id").alias("evidence_record_id"),
    F.concat(F.lit("Dry-run poll remained "), F.col("reported_state"), F.lit(" instead of "), F.col("expected_state"), F.lit(".")).alias("evidence_summary"),
)

sustainment_finding_df = gold_data_source_readiness_df.filter(F.col("required_feed_id") == "feed-sustainment").select(
    F.lit("finding-setup-sustainment-001").alias("finding_id"),
    F.lit("The sustainment feed was intentionally absent for the lesson replay.").alias("finding_title"),
    F.lit("OPEN").alias("finding_status"),
    "test_objective_id",
    F.lit("gold_data_source_readiness").alias("evidence_source_table"),
    F.col("required_feed_id").alias("evidence_record_id"),
    F.concat(F.lit("Feed status = "), F.col("required_feed_status"), F.lit(" with setup result "), F.col("setup_check_result")).alias("evidence_summary"),
)

readiness_finding_df = gold_observed_vs_predicted_df.filter((F.col("test_objective_id") == "obj-planning-readiness-001") & (F.col("system_instance_id") == "patriot-bravo-02")).select(
    F.lit("finding-setup-readiness-001").alias("finding_id"),
    F.lit("One participant remained LIMITED during the recorded setup window.").alias("finding_title"),
    F.lit("OPEN").alias("finding_status"),
    "test_objective_id",
    F.lit("gold_observed_vs_predicted").alias("evidence_source_table"),
    F.col("actual_record_id").alias("evidence_record_id"),
    F.concat(F.lit("Observed readiness = "), F.col("actual_state"), F.lit(" with health score "), F.round(F.col("actual_value"), 2).cast("string")).alias("evidence_summary"),
)

gold_finding_evidence_df = alignment_finding_df.unionByName(alignment_poll_finding_df).unionByName(sustainment_finding_df).unionByName(readiness_finding_df)

gold_tables = {
    "gold_data_source_readiness": gold_data_source_readiness_df,
    "gold_dry_run_discrepancies": gold_dry_run_discrepancies_df,
    "gold_test_objective_status": gold_test_objective_status_df,
    "gold_observed_vs_predicted": gold_observed_vs_predicted_df,
    "gold_integrated_readiness": gold_integrated_readiness_df,
    "gold_finding_evidence": gold_finding_evidence_df,
}

for table_name, frame in gold_tables.items():
    frame = add_report_labels(frame)
    (
        frame.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

print("Gold products are ready for the recorded lesson walkthrough.")

In [ ]:
WORKSPACE_ID = "10327698-2b0d-446f-9b1b-beabe18a4bda"
MIRRORED_DATABASE_ID = "fdccee22-7557-4d68-a387-aa7fc1d48343"
MIRROR_ROOT = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
    f"{MIRRORED_DATABASE_ID}/Tables/mda_ops"
)
MIRROR_SOURCE = "Azure PostgreSQL -> Fabric mirrored database"


def read_mirror_table(table_name):
    return spark.read.format("delta").load(f"{MIRROR_ROOT}/{table_name}")


mirror_site_df = read_mirror_table("site")
mirror_system_df = read_mirror_table("system_instance")
mirror_participant_df = read_mirror_table("event_participant")
mirror_test_event_df = read_mirror_table("test_event")
mirror_objective_df = read_mirror_table("test_objective")
mirror_finding_df = read_mirror_table("finding")
mirror_evidence_df = read_mirror_table("finding_evidence")
mirror_action_df = read_mirror_table("corrective_action")

lead_event_ids_df = mirror_test_event_df.filter(
    F.col("test_event_id") == "test-integrated-defense-001"
).select(
    "test_event_id",
    "scenario_id",
    "event_name",
    "planned_start_time_utc",
    "planned_end_time_utc",
)

lead_participant_df = (
    mirror_participant_df
    .join(lead_event_ids_df.select("test_event_id"), "test_event_id", "inner")
    .join(mirror_system_df, "system_instance_id", "inner")
)

site_system_summary_df = (
    lead_participant_df.groupBy("site_id")
    .agg(
        F.countDistinct("system_instance_id").alias("system_count"),
        F.sum(F.when(F.col("event_profile") != "NORMAL", 1).otherwise(0)).alias("constrained_system_count"),
        F.array_join(F.sort_array(F.collect_set("system_family")), ", ").alias("system_families"),
    )
)

gold_mirror_site_locations_df = (
    mirror_site_df.join(site_system_summary_df, "site_id", "inner")
    .select(
        "site_id",
        F.col("display_name").alias("site_name"),
        "city",
        "country_name",
        "country_code",
        "theater",
        "site_type",
        F.col("latitude").cast("double").alias("latitude"),
        F.col("longitude").cast("double").alias("longitude"),
        F.col("system_count").cast("long").alias("system_count"),
        F.col("constrained_system_count").cast("long").alias("constrained_system_count"),
        "system_families",
        "location_notice",
        "updated_at_utc",
        F.lit(MIRROR_SOURCE).alias("data_origin"),
        F.lit("mdaoperations.mda_ops.site + event_participant + system_instance").alias("mirror_source_tables"),
    )
)
assert gold_mirror_site_locations_df.filter(F.col("site_id").isNull()).count() == 0
assert gold_mirror_site_locations_df.groupBy("site_id").count().filter(F.col("count") > 1).count() == 0

active_findings_df = (
    mirror_finding_df.filter(F.col("finding_status").isin("OPEN", "IN_REVIEW"))
    .join(lead_event_ids_df, "test_event_id", "inner")
)

evidence_summary_df = (
    mirror_evidence_df.join(active_findings_df.select("finding_id"), "finding_id", "inner")
    .groupBy("finding_id")
    .agg(
        F.countDistinct("finding_evidence_id").alias("evidence_count"),
        F.array_join(F.sort_array(F.collect_set("evidence_source")), ", ").alias("evidence_sources"),
        F.min(F.struct("finding_evidence_id", "evidence_type", "evidence_reference")).alias("representative_evidence"),
        F.max("recorded_at_utc").alias("evidence_as_of_utc"),
    )
    .withColumn("representative_evidence_type", F.col("representative_evidence.evidence_type"))
    .withColumn("representative_evidence_reference", F.col("representative_evidence.evidence_reference"))
    .drop("representative_evidence")
)

action_summary_df = (
    mirror_action_df.join(active_findings_df.select("finding_id"), "finding_id", "inner")
    .groupBy("finding_id")
    .agg(
        F.countDistinct("corrective_action_id").alias("corrective_action_count"),
        F.sum(F.when(F.col("action_status") != "COMPLETE", 1).otherwise(0)).alias("open_action_count"),
        F.min(F.when(F.col("action_status") != "COMPLETE", F.struct(
            F.col("due_time_utc").isNull().alias("deadline_missing"),
            "due_time_utc",
            "corrective_action_id",
            "action_title",
            "action_status",
            "owner_role",
        ))).alias("next_action_record"),
        F.max("recorded_at_utc").alias("action_as_of_utc"),
    )
    .withColumn("next_action", F.col("next_action_record.action_title"))
    .withColumn("next_action_status", F.col("next_action_record.action_status"))
    .withColumn("next_action_owner", F.col("next_action_record.owner_role"))
    .withColumn("next_action_due_utc", F.col("next_action_record.due_time_utc"))
    .drop("next_action_record")
)

site_lookup_df = mirror_site_df.select(
    "site_id",
    F.col("display_name").alias("affected_site"),
    F.col("country_name").alias("affected_country"),
)
objective_lookup_df = mirror_objective_df.select(
    "objective_id",
    F.col("objective_name").alias("affected_objective"),
)

gold_mirror_casework_df = (
    active_findings_df
    .join(evidence_summary_df, "finding_id", "left")
    .join(action_summary_df, "finding_id", "left")
    .join(site_lookup_df, "site_id", "left")
    .join(objective_lookup_df, "objective_id", "left")
    .select(
        "scenario_id",
        "test_event_id",
        "event_name",
        "planned_start_time_utc",
        "planned_end_time_utc",
        "site_id",
        "affected_site",
        "affected_country",
        "objective_id",
        "affected_objective",
        "finding_id",
        "severity",
        "finding_status",
        "finding_title",
        "finding_summary",
        "identified_at_utc",
        F.coalesce(F.col("evidence_count"), F.lit(0)).cast("long").alias("evidence_count"),
        "evidence_sources",
        "representative_evidence_type",
        "representative_evidence_reference",
        F.coalesce(F.col("corrective_action_count"), F.lit(0)).cast("long").alias("corrective_action_count"),
        F.coalesce(F.col("open_action_count"), F.lit(0)).cast("long").alias("open_action_count"),
        "next_action",
        "next_action_status",
        "next_action_owner",
        "next_action_due_utc",
        F.greatest("identified_at_utc", "evidence_as_of_utc", "action_as_of_utc").alias("source_as_of_utc"),
        F.lit(MIRROR_SOURCE).alias("data_origin"),
        F.lit("mdaoperations.mda_ops.finding + finding_evidence + corrective_action").alias("mirror_source_tables"),
    )
)

site_risk_summary_df = (
    active_findings_df.groupBy("site_id")
    .agg(
        F.countDistinct("finding_id").alias("active_risk_count"),
        F.sum(F.when(F.col("severity") == "CRITICAL", 1).otherwise(0)).alias("critical_risk_count"),
        F.max(F.create_map(
            F.lit("LOW"), F.lit(1), F.lit("MEDIUM"), F.lit(2),
            F.lit("HIGH"), F.lit(3), F.lit("CRITICAL"), F.lit(4),
        )[F.col("severity")]).alias("risk_rank"),
    )
    .withColumn("site_risk_level", F.create_map(
        F.lit(1), F.lit("LOW"), F.lit(2), F.lit("MEDIUM"),
        F.lit(3), F.lit("HIGH"), F.lit(4), F.lit("CRITICAL"),
    )[F.col("risk_rank")])
    .drop("risk_rank")
)
gold_mirror_site_locations_df = (
    gold_mirror_site_locations_df.join(site_risk_summary_df, "site_id", "left")
    .fillna({"active_risk_count": 0, "critical_risk_count": 0, "site_risk_level": "CLEAR"})
)

mirror_gold_tables = {
    "gold_mirror_site_locations": gold_mirror_site_locations_df,
    "gold_mirror_casework": gold_mirror_casework_df,
}

for table_name, frame in mirror_gold_tables.items():
    (
        frame.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

print(f"Lead-event participating sites published: {gold_mirror_site_locations_df.count()}")
print(f"Lead-event active findings published: {gold_mirror_casework_df.count()}")

In [ ]:
spark.table("gold_data_source_readiness").orderBy("required_feed_id").show(truncate=False)
spark.table("gold_dry_run_discrepancies").orderBy("reported_time_utc").show(truncate=False)
spark.table("gold_test_objective_status").orderBy("test_objective_id").show(truncate=False)
spark.table("gold_observed_vs_predicted").orderBy("planned_record_id").show(truncate=False)
spark.table("gold_integrated_readiness").orderBy("readiness_scope", "system_instance_id").show(truncate=False)
spark.table("gold_finding_evidence").orderBy("finding_id", "evidence_record_id").show(truncate=False)
print(f"Visible quarantine rows carried forward from Silver: {silver_quarantine_df.count()}")